In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
)
# 1. Load your processed Excel file
df = pd.read_excel('default_of_credit_card_clients_processed.xlsx')

# 2. Clean infinite / NaN values from calculated ratio columns
ratio_cols = [
    'credit utilization ratio', 'pay_to_bill_ratio',
    'mean_monthly_bill', 'mean_monthly_payment', 'max_payment_delay'
]
df[ratio_cols] = df[ratio_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# 3. One-Hot Encoding for categorical features
df_encoded = pd.get_dummies(df, columns=['EDUCATION', 'Gender', 'MARRIAGE'], drop_first=True)

# 4. Feature Scaling with StandardScaler
numeric_cols_to_scale = [
    'LIMIT_BAL', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
    'mean_monthly_bill', 'mean_monthly_payment', 'credit utilization ratio', 'pay_to_bill_ratio',
    'max_payment_delay'
]
scaler = StandardScaler()
df_encoded[numeric_cols_to_scale] = scaler.fit_transform(df_encoded[numeric_cols_to_scale])

# 5. Define Features (X) and Target (y)
target_col = 'Default payment next month'
X = df_encoded.drop(columns=['ID', target_col], errors='ignore')
y = df_encoded[target_col]

# 6. Train-Test Split (80/20 with Stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 7. Train Random Forest Model with Class Balancing
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# 8. Evaluate Model Performance on Test Data
y_pred = rf_model.predict(X_test)
y_prob_test = rf_model.predict_proba(X_test)[:, 1]

print("=== ML MODEL PERFORMANCE METRICS ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_test):.4f}")

# 9. Extract Default Probability Scores for ALL Accounts
df['Default_Risk_Score'] = rf_model.predict_proba(X)[:, 1]

# 10. Assign Risk Tiers
def assign_risk_tier(score):
    if score > 0.60:
        return 'High Risk'
    elif score >= 0.30:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['Risk_Tier'] = df['Default_Risk_Score'].apply(assign_risk_tier)

print("\n=== RISK TIER DISTRIBUTION ===")
print(df['Risk_Tier'].value_counts())

# 11. Save final results with Risk Scores & Tiers for Power BI (as an Excel file)
df.to_excel('final_credit_risk_predictions.xlsx', index=False)
print("\nSuccess! Saved output to 'final_credit_risk_predictions.xlsx'")

=== ML MODEL PERFORMANCE METRICS ===
Accuracy:  0.8110
Precision: 0.6469
Recall:    0.3203
F1 Score:  0.4284
ROC-AUC:   0.7604

=== RISK TIER DISTRIBUTION ===
Risk_Tier
Low Risk       23108
High Risk       5602
Medium Risk     1290
Name: count, dtype: int64

Success! Saved output to 'final_credit_risk_predictions.xlsx'


In [ ]:
from google.colab import files
files.download('final_credit_risk_predictions.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>